# Pipeline LangGraph — Caso 3

**Objetivo:** orquestar el flujo completo de decisión: carga → features → reglas → LLM → decisión final.

**Arquitectura:**

```
load_case → compute_features → apply_rules
                                    │
                        ┌───────────┼───────────┐
                        ▼           ▼           ▼
                    RECHAZAR    APROBAR     ambiguo
                        │           │           │
                        ▼           ▼     llm_classify
                   final_decision   │           │
                        │           │     final_decision
                        ▼           ▼           │
                   generate_output ◄────────────┘
                        │
                       END
```

**Modelos:**
- Reglas heurísticas: `src/rules/rule_engine.py` (thresholds en `thresholds.yaml`)
- LLM: OpenRouter `MODEL_FRAUD` para casos ambiguos

**Input:** PostgreSQL `casos` (150 originales + 100 sintéticos)
**Output:** recomendación + justificación + señales + decisión

## 1. Setup y compilación del grafo

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import sys
from pathlib import Path

# Asegurar que el proyecto esté en el path
PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

from src.pipeline.graph import build_graph

graph = build_graph()
print('[OK] Grafo LangGraph compilado')

[OK] Grafo LangGraph compilado


## 2. Ejemplo: caso RECHAZADO por reglas

Un caso con flags de fraude previos ≥ 2 no necesita LLM — las reglas lo rechazan directo.

In [2]:
result = await graph.ainvoke({'case_id': 'COMP-0009'})
print(f'Caso: {result["case_id"]}')
print(f'Decisión: {result["final_decision"]} (decisión: {result.get("rule_disparada", "sin regla")})')
print(f'Justificación: {result["justification"]}')
print(f'Señales: {result["signals"]}')

Caso: COMP-0009
Decisión: RECHAZAR (decisión: sin regla)
Justificación: RECHAZAR por señales de fraude detectadas por reglas: R1: flags_fraude_previos >= 2.
Señales: ['R1: flags_fraude_previos >= 2']


## 3. Ejemplo: caso APROBADO por reglas

Un caso con GPS confirmada, usuario antiguo y ratio bajo se aprueba directo.

In [3]:
result = await graph.ainvoke({'case_id': 'COMP-0012'})
print(f'Caso: {result["case_id"]}')
print(f'Decisión: {result["final_decision"]} (decisión: {result.get("rule_disparada", "sin regla")})')
print(f'Justificación: {result["justification"]}')
print(f'Señales: {result["signals"]}')

Caso: COMP-0012
Decisión: APROBAR (decisión: sin regla)
Justificación: APROBAR: caso consistente con perfil legítimo. Señales: A3: GPS confirmada con usuario antiguo.
Señales: ['A3: GPS confirmada con usuario antiguo']


## 4. Ejemplo: caso ambiguo → LLM

Cuando las reglas no son concluyentes, el caso pasa al LLM para análisis del texto.

In [4]:
result = await graph.ainvoke({'case_id': 'COMP-0001'})
print(f'Caso: {result["case_id"]}')
print(f'Decisión: {result["final_decision"]} (decisión: {result.get("rule_disparada", "sin regla")})')
print(f'Justificación: {result["justification"]}')
if result.get('llm_analysis'):
    print(f'LLM veredicto: {result["llm_analysis"].get("veredicto", "N/A")}')
    print(f'LLM justificación: {result["llm_analysis"]["justificacion"][:200]}')

Caso: COMP-0001
Decisión: APROBAR (decisión: sin regla)
Justificación: APROBAR por análisis LLM: Usuario con 1590 días de antigüedad y 0 flags de fraude. Reclamo por error de restaurante sin evidencia GPS que lo contradiga. Frecuencia, monto y ratio de compensación dentro de parámetros normales.
LLM veredicto: APROBAR: reclamo coherente con perfil de usuario sano y sin señales de fraude o inconsistencia.
LLM justificación: Usuario con 1590 días de antigüedad y 0 flags de fraude. Reclamo por error de restaurante sin evidencia GPS que lo contradiga. Frecuencia, monto y ratio de compensación dentro de parámetros normales.


## 5. Resultados del batch completo (250 casos)

Distribución final después de procesar todos los casos.

In [5]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host='localhost', port=5432, dbname='rappi_cases',
    user='rappi', password='rappi_pass'
)

def _consulta(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print('--- Distribución total ---')
df_dist = _consulta(
    'SELECT decision, COUNT(*) AS n FROM analisis_casos '
    'GROUP BY decision ORDER BY n DESC'
)
df_dist['pct'] = (df_dist['n'] / df_dist['n'].sum() * 100).round(1)
print(df_dist.to_string(index=False))

print('\n--- Por origen (original vs sintético) ---')
df_origen = _consulta(
    'SELECT c.es_sintetico, a.decision, COUNT(*) AS n '
    'FROM casos c JOIN analisis_casos a ON a.caso_id = c.caso_id '
    'GROUP BY c.es_sintetico, a.decision ORDER BY 1, 3 DESC'
)
print(df_origen.to_string(index=False))

conn.close()

--- Distribución total ---
decision   n  pct
 ESCALAR 113 45.2
RECHAZAR  88 35.2
 APROBAR  49 19.6

--- Por origen (original vs sintético) ---
 es_sintetico decision  n
        False  ESCALAR 80
        False RECHAZAR 50
        False  APROBAR 20
         True RECHAZAR 38
         True  ESCALAR 33
         True  APROBAR 29


### Lectura

- **35.2% RECHAZADOS:** el sistema detecta fraude con alta certeza por señales claras (flags, inconsistencia GPS, abuso).
- **19.6% APROBADOS:** casos claramente legítimos se automatican sin pasar por LLM.
- **45.2% ESCALADOS:** casos ambiguos van a revisión humana con contexto.

La alta proporción de ESCALAR es intencional: el sistema **no fuerza decisiones binarias** donde no las hay. Los casos escalados incluyen el contexto ya procesado (features, señales, análisis LLM) para que el agente humano decida rápido.

## 6. Verificación

Confirmamos que todos los casos tienen decisión y no hay nulos.

In [6]:
conn = psycopg2.connect(
    host='localhost', port=5432, dbname='rappi_cases',
    user='rappi', password='rappi_pass'
)

with conn.cursor() as cur:
    cur.execute(
        '''SELECT
             COUNT(*) AS total,
             COUNT(a.decision) AS con_decision,
             COUNT(a.justificacion) AS con_justificacion,
             COUNT(a.senales_usadas) AS con_senales
           FROM casos c LEFT JOIN analisis_casos a ON a.caso_id = c.caso_id'''
    )
    cols = [d[0] for d in cur.description]
    check = pd.DataFrame(cur.fetchall(), columns=cols)
conn.close()

print(check.to_string(index=False))
row = check.iloc[0]
assert row['total'] == row['con_decision'] == 250
assert row['con_justificacion'] == 250
assert row['con_senales'] == 250
print('\n[OK] Step 06 completo: pipeline ejecutado sobre 250 casos.')

 total  con_decision  con_justificacion  con_senales
   250           250                250          250

[OK] Step 06 completo: pipeline ejecutado sobre 250 casos.
